# Assignment 1: EDA — Road Pothole Severity Assessment

**Student:** Jayanan M  
**Branch:** Computer Science and Engineering  

## Objective
Explore road-pothole detection records, check data quality, visualize patterns, and assess factors related to pothole severity.

> **Dataset note:** The CSV included with this assignment is a reproducible **synthetic educational dataset** created from the documented structure and severity bands of a published pothole field-test log. It is not the original field-test CSV.

## 1. Dataset source and context

The public IPDS pothole-detection project documents a sample field-test log containing detection confidence, bounding-box area, aspect ratio, peak jerk, severity and GPS fields.

Its published sample reports Low 16%, Medium 46%, High 38%, with peak-jerk bands of 1.6–2.7, 3.0–5.9 and 6.0–9.4.

For this academic EDA, a larger reproducible teaching dataset was generated using those documented ranges. This keeps the workflow complete without misrepresenting generated rows as original observations.

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")

df = pd.read_csv("data/road_pothole_severity_eda.csv")
print("Dataset loaded successfully.")
print("Shape:", df.shape)
display(df.head())

## 2. Initial dataset exploration

In [ ]:
display(df.head())
display(df.tail())
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
df.info()
display(df.describe(include="all").T)

## 3. Data quality analysis

In [ ]:
missing_summary = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2)
}).sort_values("missing_percent", ascending=False)

display(missing_summary)
print("Duplicate rows:", df.duplicated().sum())
display(df["severity"].value_counts(dropna=False))

In [ ]:
missing = missing_summary[missing_summary["missing_count"] > 0]
plt.figure(figsize=(10, 5))
plt.bar(missing.index.astype(str), missing["missing_count"])
plt.title("Missing Values by Column")
plt.xlabel("Column")
plt.ylabel("Missing Values")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

## 4. Data cleaning and preprocessing

In [ ]:
before = len(df)
df_clean = df.drop_duplicates().copy()

for col in ["confidence", "aspect_ratio"]:
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())

print("Duplicate rows removed:", before - len(df_clean))
display(df_clean.isna().sum().to_frame("missing_count"))
print("Cleaned shape:", df_clean.shape)

## 5. Invalid values and IQR outlier analysis

In [ ]:
numeric_cols = ["confidence", "bounding_box_area", "aspect_ratio", "peak_jerk"]

for col in numeric_cols:
    print(f"{col}: {(df_clean[col] < 0).sum()} negative values")

q1 = df_clean[numeric_cols].quantile(0.25)
q3 = df_clean[numeric_cols].quantile(0.75)
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr
mask = df_clean[numeric_cols].lt(lower) | df_clean[numeric_cols].gt(upper)

outliers = pd.DataFrame({
    "outlier_count": mask.sum(),
    "outlier_percent": (mask.mean()*100).round(2),
    "lower_bound": lower.round(3),
    "upper_bound": upper.round(3)
})
display(outliers)
print("IQR flags unusual values; it does not prove those observations are errors.")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, col in zip(axes.ravel(), numeric_cols):
    ax.boxplot(df_clean[col].dropna(), vert=False)
    ax.set_title(f"Box Plot: {col}")
plt.tight_layout()
plt.show()

## 6. Statistical analysis

In [ ]:
stats = df_clean[numeric_cols].agg(
    ["count", "mean", "median", "std", "min", "max"]
).T.round(3)
display(stats)

## 7. Univariate analysis

In [ ]:
df_clean[numeric_cols].hist(figsize=(12, 9), bins=20, edgecolor="black")
plt.suptitle("Distributions of Numerical Features", y=1.02)
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 5))
sns.countplot(data=df_clean, x="severity", order=["Low", "Medium", "High"])
plt.title("Pothole Severity Distribution")
plt.xlabel("Severity")
plt.ylabel("Number of Records")
plt.tight_layout()
plt.show()

severity_summary = df_clean["severity"].value_counts().reindex(["Low", "Medium", "High"])
display(pd.DataFrame({
    "count": severity_summary,
    "percentage": (severity_summary / len(df_clean) * 100).round(2)
}))

## 8. Bivariate analysis and correlation heatmap

In [ ]:
corr = df_clean[numeric_cols].corr()
plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt=".2f", center=0)
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()
display(corr["peak_jerk"].drop("peak_jerk").sort_values(key=np.abs, ascending=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for i, col in enumerate(["peak_jerk", "bounding_box_area"]):
    data = [df_clean.loc[df_clean["severity"] == s, col].dropna()
            for s in ["Low", "Medium", "High"]]
    axes[i].boxplot(data, tick_labels=["Low", "Medium", "High"])
    axes[i].set_title(f"{col.replace('_',' ').title()} by Severity")
    axes[i].set_xlabel("Severity")
axes[0].set_ylabel("Peak Jerk (m/s³)")
axes[1].set_ylabel("Bounding Box Area (px²)")
plt.tight_layout()
plt.show()

## 9. Multivariate analysis

In [ ]:
pair_cols = ["confidence", "bounding_box_area", "aspect_ratio", "peak_jerk"]
pair_df = df_clean[pair_cols + ["severity"]].dropna()
sns.pairplot(pair_df.sample(min(250, len(pair_df)), random_state=42),
             hue="severity", hue_order=["Low", "Medium", "High"], diag_kind="hist")
plt.suptitle("Multivariate Pair Plot", y=1.02)
plt.show()

## 10. Findings and interpretation

### Dataset and data quality
- Initial dataset size: **303 rows × 9 columns**.
- Duplicate rows identified: **3**.
- Cleaned dataset size: **300 rows**.
- Missing numerical values were handled using median imputation for EDA.
- No negative values were found in the numerical measurement fields.

### Severity distribution
- Low: **48 (16.0%)**
- Medium: **138 (46.0%)**
- High: **114 (38.0%)**

### Relationships
- Peak jerk is clearly separated across severity groups because the teaching data follows the documented severity bands.
- The strongest absolute association with peak jerk among the other numerical variables is **bounding_box_area**, with **r = 0.88**.
- Correlation indicates association, not causation.

### Outliers
IQR analysis was used to flag unusual values. The observations were retained because an IQR flag does not automatically mean an observation is erroneous.

### Limitations
The CSV is synthetic and intended for academic EDA practice. It must not be presented as the original 50 field-test observations or as new real-world road-safety evidence.

## 11. Conclusion

This project demonstrates a complete EDA workflow for road-pothole severity data: data inspection, data-quality checking, duplicate removal, missing-value handling, outlier analysis, descriptive statistics, univariate analysis, bivariate correlation analysis, severity comparison and multivariate visualization.

The project is structured for academic submission and GitHub upload. The synthetic-data limitation is explicitly documented so that generated observations are not mistaken for original field measurements.

## 12. Submission checklist

- [x] Dataset context and source documented
- [x] head(), tail(), shape, info(), describe() included
- [x] Missing values checked
- [x] Duplicates checked and handled
- [x] Invalid values checked
- [x] Outliers analyzed
- [x] Cleaning/preprocessing completed
- [x] Statistical analysis completed
- [x] Univariate analysis completed
- [x] Bivariate analysis completed
- [x] Correlation heatmap included
- [x] Multivariate pair plot included
- [x] Findings and conclusion completed
- [x] Limitations documented